# SrTiO$_3$: Wannier interpolation and Berry-curvature checks

Meeting summary, in the order used to validate the calculation. All Born-charge and curvature results below use the production **`mmn_pair`** construction: right-orbital phase factors in the `.mmn` overlaps, orbital-pair-dependent real-space vectors for both $A(\mathbf R)$ and $H(\mathbf R)$, and one shared Berry-curvature implementation.

The position matrix was tested using two constructions. The direct **`_r.dat` route** reads Wannier90's matrix elements $\langle 0i|\hat r_\alpha|\mathbf Rj\rangle$. 

The **`.mmn` route** instead reconstructs the Wannier-gauge connection from the finite-difference overlaps,

$$A_\alpha(\mathbf k)=i\sum_{\mathbf b}w_{\mathbf b}b_\alpha\,U_{\mathbf k}^\dagger M^{(\mathbf k,\mathbf b)}U_{\mathbf k+\mathbf b}\,e^{i\mathbf b\cdot\boldsymbol\tau_j},$$

and then Fourier transforms it to $A_\alpha(\mathbf R)$. The last phase is tied to the right orbital $j$ and retains its intracell position.

A finite source mesh fixes each real-space matrix only up to supercell translations of $\mathbf R$. The conventional Wannier90 **`_hr.dat`/`_r.dat` choice minimizes $|\mathbf R|$**, independently of the orbital pair. Here both $H_{ij}(\mathbf R)$ and $A_{ij}(\mathbf R)$ instead use the representative minimizing the physical separation

$$|\mathbf R+\boldsymbol\tau_j-\boldsymbol\tau_i|.$$

This pair-dependent choice follows the actual Wannier-center-to-center bond and is covariant under the crystal symmetries. Using the same rule for $H$ and $A$ is essential: mixing the two $\mathbf R$ conventions agrees on the source mesh but does not define a consistent interpolant between its points.

**Ti-$z$ displacement:** $+0.01$ Å, models (a) 60 WFs / 30 occupied, (b) occupied-only 30 WFs, and (c) 48 WFs / 38 occupied.  
**Sr-$z$ displacement:** $+0.01$ Å, models (b) and (c).  
**DFPT references:** $Z^*_{\mathrm{Ti},zz}=7.28829$ and $Z^*_{\mathrm{Sr},zz}=2.55488$.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modules/axion.py").exists())
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def read_band_blocks(path):
    data = np.loadtxt(path)
    cuts = np.r_[0, np.flatnonzero(np.diff(data[:, 0]) < 0) + 1, len(data)]
    return [data[cuts[i]:cuts[i + 1]] for i in range(len(cuts) - 1)]

RESULTS = ROOT / "validation/results/convention_symmetry"
born = pd.read_csv(RESULTS / "born_charge_comparison.csv")
sym = np.load(RESULTS / "symmetry_curves.npz")
sr_born_b = pd.read_csv(DATA / "SrTiO3/Sr_Q_01A/output/184367bc889/trial_02_Sr_p_O_sp/no_displacement/188438bc889" / (
    "born_charge/"
    "born_charge_Sr_Q_01A_mmn_taucentres_wbnone.csv"
))
sr_born_c = pd.read_csv(DATA / "SrTiO3/Sr_Q_01A/output/184367bc889/trial_03_Sr_sp_Ti_pd_O_sp/no_displacement/188444bc889" / (
    "born_charge/"
    "born_charge_Sr_Q_01A_mmn_taucentres_mhn0_wbnone.csv"
))

trials = ["a", "b", "c"]
trial_names = {
    "a": "(a) 60 WFs / 30 occ.",
    "b": "(b) 30 WFs / occupied-only",
    "c": "(c) 48 WFs / 38 occ.",
}
band_titles = {
    "a": r"Set (a): Sr $s,p,d$; Ti $s,p,d$; O $s,p$",
    "b": r"Set (b): Sr $p$; O $s,p$",
    "c": r"Set (c): Sr $s,p$; Ti $p,d$; O $s,p$",
}
colors = {"a": "#d62728", "b": "#1f77b4", "c": "#2ca02c"}

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.size": 10,
})
print(f"Loaded compact validation data from {RESULTS}")

## 1. Wannier-interpolated bands and energy windows

Separate base-structure band figures are shown for sets (a), (b), and (c). Their trial orbitals are (a) Sr $s,p,d$; Ti $s,p,d$; O $s,p$; (b) Sr $p$; O $s,p$; and (c) Sr $s,p$; Ti $p,d$; O $s,p$. Each figure includes the full QE energy range, its own Wannier90 outer and frozen windows, and the Fermi level $E_F=10.0561$ eV.

$$
H_{mn}(\mathbf k)=\sum_{\mathbf R}e^{2\pi i\mathbf k\cdot(\mathbf R+\boldsymbol\tau_n-\boldsymbol\tau_m)}H_{mn}(\mathbf R).
$$

The stored QE band reference under `base/soc` is byte-identical to the Ti-$z$ $+0.01$ Å artifact, so it is a common near-base reference rather than an independent second band run.

In [ ]:
base = DATA / "SrTiO3/base/soc/output/181642bc889"
qe_path = next(base.glob("bands/*/SrTiO3_bands.dat.gnu"))
band_models = {
    "a": DATA / "SrTiO3/base/soc/output/181642bc889/trial_01_Sr_spd_Ti_spd_O_sp/182494bc889",
    "b": DATA / "SrTiO3/base/soc/output/181642bc889/trial_02_Sr_p_O_sp/183813bc889",
    "c": DATA / "SrTiO3/base/soc/output/181642bc889/trial_03_Sr_sp_Ti_pd_O_sp/185116bc889",
}

qe_bands = read_band_blocks(qe_path)
e_fermi = 11
reference_bands = read_band_blocks(band_models["a"] / "SrTiO3_band.dat")
qe_scale = reference_bands[0][-1, 0] / qe_bands[0][-1, 0]

labels, nodes = [], []
for line in (band_models["a"] / "SrTiO3_band.labelinfo.dat").read_text().splitlines():
    fields = line.split()
    labels.append(r"$\Gamma$" if fields[0] == "G" else fields[0])
    nodes.append(float(fields[2]))

for trial in trials:
    directory = band_models[trial]
    w90_bands = read_band_blocks(directory / "SrTiO3_band.dat")
    window = {}
    for line in (directory / "SrTiO3.win").read_text().splitlines():
        if "=" in line:
            key, value = (field.strip() for field in line.split("=", 1))
            if key in {"dis_win_min", "dis_win_max", "dis_froz_min", "dis_froz_max"}:
                window[key] = float(value)

    fig, ax = plt.subplots(figsize=(8.4, 5.2))
    for i, band in enumerate(qe_bands):
        ax.plot(qe_scale * band[:, 0], band[:, 1], color="0.18", lw=0.68,
                alpha=0.46, label="QE" if i == 0 else None)
    for i, band in enumerate(w90_bands):
        ax.plot(band[:, 0], band[:, 1], color=colors[trial], lw=0.95,
                alpha=0.90, label="Wannier" if i == 0 else None)
    for x in nodes:
        ax.axvline(x, color="0.80", lw=0.65, zorder=0)

    ax.axhline(e_fermi, color="black", lw=1.0, label=r"$E_F$")
    outer = (window["dis_win_min"], window["dis_win_max"])
    frozen = (window["dis_froz_min"], window["dis_froz_max"])
    for index, energy in enumerate(outer):
        ax.axhline(energy, color="#9467bd", ls="--", lw=1.2,
                   label="outer window" if index == 0 else None)
    if frozen != outer:
        for index, energy in enumerate(frozen):
            ax.axhline(energy, color="#1f77b4", ls=":", lw=1.4,
                       label="frozen window" if index == 0 else None)
    else:
        ax.plot([], [], color="#9467bd", ls="--", lw=1.2,
                label="outer = frozen")

    ax.set(xlim=(nodes[0], nodes[-1]), ylim=(-50, 42), ylabel=r"$E$ (eV)",
           xticks=nodes, xticklabels=labels, title=band_titles[trial])
    ax.legend(frameon=False, bbox_to_anchor=(1.01, 1.0), loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()

## 2. DFPT versus interpolated Born charges

The electronic response is the Brillouin-zone integral of the mixed curvature,

$$
Z^*_{\kappa,\alpha\beta}=Z^{\rm ion}_{\kappa}\delta_{\alpha\beta}
-\frac{eV}{(2\pi)^3}\int_{\rm BZ}d^3k\;\mathrm{Tr}\,\Omega_{k_\alpha u_{\kappa\beta}}(\mathbf k),
$$

with $Z^{\rm el}$ decomposed through $\Omega=\Omega^{\rm int}+\Omega^{\rm cross}+\Omega^{\rm ext}$. Because this curvature covers only the occupied states retained in the Wannier window, excluded occupied semicore states that move rigidly with atom $\kappa$ are folded into an effective ionic term,

$$
Z^{\rm ion,eff}_{\kappa}=z^{\rm PP}_{\kappa}-N^{\rm out}_{\kappa},
\qquad Z^*_{\kappa,zz}=Z^{\rm ion,eff}_{\kappa}+Z^{\rm el}_{\kappa,zz}.
$$

Here $z^{\rm PP}_{\mathrm{Ti}}=12$ and $z^{\rm PP}_{\mathrm{Sr}}=10$. Thus $Z^{\rm ion,eff}_{\mathrm{Ti}}=(4,4,10)$ for sets (a,b,c), while $Z^{\rm ion,eff}_{\mathrm{Sr}}=(8,10)$ for sets (b,c). These set-dependent values are Wannier-window bookkeeping—not different physical ionic charges—and their change is compensated by the electronic term when the retained subspace is accurate.

In [ ]:
converged = born.query("convention == 'mmn_pair' and nk == 16").copy()
converged["model"] = converged["trial"].map(trial_names)
converged["electronic"] = converged[["z_int_zz", "z_cross_zz", "z_ext_zz"]].sum(axis=1)
converged["ionic"] = converged["z_total_zz"] - converged["electronic"]
ti_table = converged[["model", "z_int_zz", "z_cross_zz", "z_ext_zz", "electronic", "ionic", "z_total_zz", "zz_relative_percent"]]
ti_table.columns = ["Ti model", "internal", "cross", "external", "electronic", "ionic", "Z*_zz", "error (%)"]
print("Ti-z displacement (+0.01 A)")
print(ti_table.to_string(index=False, formatters={c: "{:.6f}".format for c in ti_table.columns[1:]}))

sr_rows = []
for trial, data in (("b", sr_born_b), ("c", sr_born_c)):
    row = data.query("nk == 16").iloc[0]
    sr_rows.append({
        "trial": trial,
        "model": trial_names[trial],
        "internal": row["z_int_z"],
        "cross": row["z_cross_z"],
        "external": row["z_ext_z"],
        "electronic": row["z_el_z"],
        "ionic": row["z_total_z"] - row["z_el_z"],
        "Z*_zz": row["z_total_z"],
        "DFPT": row["z_qe_z"],
        "error (%)": 100 * (row["z_total_z"] - row["z_qe_z"]) / row["z_qe_z"],
    })
sr_converged = pd.DataFrame(sr_rows)
print("\nSr-z displacement (+0.01 A)")
print(sr_converged[["model", "internal", "cross", "external", "electronic", "ionic", "Z*_zz", "error (%)"]].to_string(
    index=False, formatters={c: "{:.6f}".format for c in ("internal", "cross", "external", "electronic", "ionic", "Z*_zz", "error (%)")}
))

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), gridspec_kw={"width_ratios": [1.7, 1]})
ax = axes[0]
x = np.arange(len(converged))
bars = ax.bar(x, converged["z_total_zz"], color=[colors[t] for t in converged["trial"]], width=0.62)
ax.axhline(converged["z_qe_zz"].iloc[0], color="black", lw=1.3, label="DFPT = 7.28829")
for bar, value, err in zip(bars, converged["z_total_zz"], converged["zz_relative_percent"]):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.012,
            f"{value:.4f}\n({err:.3f}%)", ha="center", va="bottom", fontsize=8.5)
ax.set(xticks=x, xticklabels=["(a)", "(b)", "(c)"], ylabel=r"$Z^*_{zz}$",
       ylim=(7.24, 7.49), title=r"Ti $+0.01$ Å")
ax.legend(frameon=False, loc="upper right", fontsize=9)

ax = axes[1]
x = np.arange(len(sr_converged))
bars = ax.bar(x, sr_converged["Z*_zz"],
              color=[colors[t] for t in sr_converged["trial"]], width=0.58)
ax.axhline(sr_converged["DFPT"].iloc[0], color="black", lw=1.3, label="DFPT = 2.55488")
for bar, value, err in zip(bars, sr_converged["Z*_zz"], sr_converged["error (%)"]):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.003,
            f"{value:.4f}\n({err:+.3f}%)", ha="center", va="bottom", fontsize=8.5)
ax.set(xticks=x, xticklabels=["(b)", "(c)"], ylim=(2.50, 2.58), title=r"Sr $+0.01$ Å")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

## 3. Born charge versus $n_k$

The left panel shows all three Ti-displacement models; the right panel shows Sr-displacement models (b) and (c). This tests interpolation-grid quadrature only. The underlying Wannier data were generated on an $8^3$ mesh.

In [ ]:
prod = born.query("convention == 'mmn_pair'")
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.1), gridspec_kw={"width_ratios": [1.7, 1]})

ax = axes[0]
for trial in trials:
    rows = prod[prod["trial"] == trial].sort_values("nk")
    ax.plot(rows["nk"], rows["z_total_zz"], "o-", lw=1.8, ms=4.5,
            color=colors[trial], label=trial_names[trial])
ax.axhline(prod["z_qe_zz"].iloc[0], color="black", lw=1.2, label="DFPT")
ax.set(xlabel=r"interpolation mesh $n_k$", ylabel=r"$Z^*_{zz}$",
       xticks=sorted(prod["nk"].unique()), title=r"Ti $+0.01$ Å")
ax.legend(frameon=False, fontsize=8.3)

ax = axes[1]
for trial, data in (("b", sr_born_b), ("c", sr_born_c)):
    ax.plot(data["nk"], data["z_total_z"], "o-", lw=1.8, ms=4.5,
            color=colors[trial], label=trial_names[trial])
ax.axhline(sr_born_b["z_qe_z"].iloc[0], color="black", lw=1.2, label="DFPT")
ax.set(xlabel=r"interpolation mesh $n_k$", xticks=sr_born_b["nk"].tolist(),
       title=r"Sr $+0.01$ Å")
ax.legend(frameon=False, fontsize=8.0)
plt.tight_layout()
plt.show()

## 4. Base structure: $PT$ enforces $\mathrm{Tr}\,\Omega_{xy}=0$

Berry curvature is even under inversion and odd under time reversal:

$$
P:\Omega_{xy}(\mathbf k)=\Omega_{xy}(-\mathbf k),\qquad
T:\Omega_{xy}(\mathbf k)=-\Omega_{xy}(-\mathbf k).
$$

Therefore the occupied trace must vanish pointwise when both symmetries are present. The test path is $\mathbf k(t)=(t,0.30,0.15)$.

In [ ]:
t = sym["t"]
fig, ax = plt.subplots(figsize=(7.4, 4.0))
for trial in trials:
    values = sym[f"{trial}__base__mmn_pair__generic"]
    ax.plot(t, values, lw=1.7, color=colors[trial], label=trial_names[trial])
ax.axhline(0, color="black", lw=0.9)
ax.set(xlabel=r"$t$ in $\mathbf{k}(t)=(t,0.30,0.15)$",
       ylabel=r"$\mathrm{Tr}\,\Omega_{xy}$ (Å$^2$)",
       title=r"Undisplaced structure: pointwise $PT$ zero")
ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

for trial in trials:
    maximum = np.max(np.abs(sym[f"{trial}__base__mmn_pair__generic"]))
    print(f"{trial_names[trial]:30s} max |Tr Omega_xy| = {maximum:.3e} A^2")

## 5. Displaced structures: mirror-plane zeros

Both the Ti-$z$ and Sr-$z$ displacements preserve the mirror $M_x$. Berry curvature transforms as an axial vector, so $\Omega_{xy}\equiv\Omega_z$ changes sign under $M_x$ and must vanish on the invariant plane $k_x=\tfrac12$. Separate figures are shown for $\mathbf k(t)=(0.50,t,0.15)$ in each displaced structure.

In [ ]:
mirror_cases = [
    ("Ti-z displaced structure", [
        (trial_names[trial], colors[trial], sym[f"{trial}__mode__mmn_pair__mirror_kx_half"])
        for trial in trials
    ]),
    ("Sr-z displaced structure", [
        (trial_names["b"], colors["b"], sym["sr_b__mode__mmn_pair__mirror_kx_half"]),
        (trial_names["c"], colors["c"], sym["sr_c__mode__mmn_pair__mirror_kx_half"]),
    ]),
]

for structure_name, curves in mirror_cases:
    fig, ax = plt.subplots(figsize=(7.4, 4.0))
    for label, color, values in curves:
        ax.plot(t, values, lw=1.7, color=color, label=label)
    ax.axhline(0, color="black", lw=0.9)
    ax.set(xlabel=r"$t$ in $\mathbf{k}(t)=(0.50,t,0.15)$",
           ylabel=r"$\mathrm{Tr}\,\Omega_{xy}$ (Å$^2$)",
           title=structure_name + r": $k_x=1/2$ mirror plane")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()

    for label, _, values in curves:
        print(f"{structure_name:26s} {label:30s} max |Tr Omega_xy| = "
              f"{np.max(np.abs(values)):.3e} A^2")

## 6. Displaced structures: zeros at mirror intersections

Away from a mirror plane, each displaced crystal has a finite physical curvature. The generic path $\mathbf k(t)=(t,0.30,0.15)$ intersects mirror-invariant planes at

$$t=0,\;0.30,\;0.50,\;0.70,\;1,$$

where $\mathrm{Tr}\,\Omega_{xy}$ must vanish. Ti-$z$ and Sr-$z$ results are plotted separately.

In [ ]:
zeros = sym["forced_zeros"]
zero_indices = [int(np.argmin(np.abs(t - z))) for z in zeros]
generic_cases = [
    ("Ti-z displaced structure", [
        (trial_names[trial], colors[trial], sym[f"{trial}__mode__mmn_pair__generic"])
        for trial in trials
    ]),
    ("Sr-z displaced structure", [
        (trial_names["b"], colors["b"], sym["sr_b__mode__mmn_pair__generic"]),
        (trial_names["c"], colors["c"], sym["sr_c__mode__mmn_pair__generic"]),
    ]),
]

for structure_name, curves in generic_cases:
    fig, ax = plt.subplots(figsize=(8.0, 4.3))
    for label, color, values in curves:
        ax.plot(t, values, lw=1.9, color=color, label=label)
        ax.scatter(zeros, values[zero_indices], s=28, color=color, zorder=4)
    for z in zeros:
        ax.axvline(z, color="0.65", ls=":", lw=0.9)
    ax.axhline(0, color="black", lw=0.9)
    ax.set(xlabel=r"$t$ in $\mathbf{k}(t)=(t,0.30,0.15)$",
           ylabel=r"$\mathrm{Tr}\,\Omega_{xy}$ (Å$^2$)",
           title=structure_name + ": mirror intersections")
    ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()

    for label, _, values in curves:
        forced = np.max(np.abs(values[zero_indices]))
        physical = np.max(np.abs(values))
        print(f"{structure_name:26s} {label:30s} forced-zero max = "
              f"{forced:.3e} A^2; off-symmetry scale = {physical:.3e} A^2")

## Takeaway

- The effective ionic contribution is set by the displaced atom and the Wannier window: Ti (a,b,c) uses $(+4,+4,+10)$ and Sr (b,c) uses $(+8,+10)$. The remainder is the in-window electronic curvature response.
- The Ti-displaced 48-WF model (c) gives $Z^*_{\mathrm{Ti},zz}=7.30099$, **0.174%** above DFPT.
- Sr model (b) gives $Z^*_{\mathrm{Sr},zz}=2.54299$ (**0.465% below** DFPT); model (c) improves this to $2.55120$ (**0.144% below** DFPT).
- All Ti and Sr $n_k$ sequences are stable by $12^3$–$16^3$.
- Ti-$z$: all production symmetry zeros are $\leq 8.1\times10^{-8}$ Å$^2$, versus off-symmetry signals up to $2.40\times10^{-2}$ Å$^2$.
- Sr-$z$: models (b,c) have symmetry-zero residuals $\leq 8.4\times10^{-9}$ Å$^2$, versus off-symmetry signals of $2.79\times10^{-3}$ and $5.93\times10^{-3}$ Å$^2$.

The remaining systematic limits are the $8^3$ source Wannier mesh, a one-sided $0.01$ Å displacement, and the frozen-Wannier-gauge approximation $A_\beta=0$ because $\Lambda(\mathbf R)=\langle 0s|\partial_\beta|\mathbf Rt\rangle$ is unavailable.